In [20]:
# install langgraph v1.0 + groq - modern 2026 stack
!pip install langgraph langchain-groq langchain-core -q

In [38]:
# import graph building tools - same as v1
from langgraph.graph import StateGraph, START, END

# MessagesState = modern built-in state - holds full chat list, not just one string
from langgraph.graph import MessagesState

# MemorySaver = robot's memory - saves everything after each node runs
from langgraph.checkpoint.memory import MemorySaver

# ChatGroq = the brain we plug into our worker node
from langchain_groq import ChatGroq

# HumanMessage = wraps your question like putting it in an envelope
from langchain_core.messages import HumanMessage

In [58]:
from google.colab import userdata
import os

# getting key from colab secret locker - never exposed in code
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
from langchain_groq import ChatGroq

# Re-initializing the brain to ensure it picks up the latest API key
llm = ChatGroq(model="llama-3.1-8b-instant")

In [59]:
from google.colab import userdata
import os

# getting key from colab secret locker - safe, never shows in code
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [60]:
# our thinking worker - takes chat history, sends to brain, gets reply
def thinking_node(state: MessagesState):   # state = whatsapp chat list

    response = llm.invoke(state["messages"])  # send full chat to groq brain → get reply

    return {"messages": response}  # add reply back to chat list

In [61]:
# step 1 - create graph with MessagesState (whatsapp chat list as state)
graph = StateGraph(MessagesState)

# step 2 - register our thinking worker
graph.add_node("thinking_node", thinking_node)

# step 3 - connect roads
graph.add_edge(START, "thinking_node")   # start → thinking worker
graph.add_edge("thinking_node", END)     # thinking worker → done

# step 4 - create memory
memory = MemorySaver()   # this is the robot's memory box

# step 5 - compile WITH memory plugged in
app = graph.compile(checkpointer=memory)  # memory attached from day 1

In [64]:
config = {"configurable": {"thread_id": "chat1"}}  # which chat group to save to

result = app.invoke(
    {"messages": [HumanMessage(content="What is AI?")]},
    config=config   # tell memory which group to save this to
)

print(result["messages"][-1].content)  # print last message = robot's reply

You just asked me the same question again. I already provided a detailed explanation of AI earlier. If you'd like, I can summarize it for you:

AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It's based on algorithms that allow machines to analyze data, recognize patterns, and make decisions with minimal human intervention.

Would you like to know more about a specific aspect of AI or is there something else I can help you with?


In [63]:
# same config = same locker = robot remembers previous conversation
result2 = app.invoke(
    {"messages": [HumanMessage(content="What did I just ask you?")]},
    config=config  # same chat1 locker
)

print(result2["messages"][-1].content)

You just asked me "What is AI?"
